In [10]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


data = pd.read_csv("../data/processed_data.csv")

data = pd.get_dummies(data,columns=["Type", "Store", "IsHoliday"],drop_first=True)


X = data.drop("Weekly_Sales", axis=1)
y = data["Weekly_Sales"]

# Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)



# XGBoost Model
xg_model = XGBRegressor(objective="reg:squarederror",random_state=0,n_jobs=1)

# Parameters
params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# Randomized Search
xgb_search = RandomizedSearchCV(
    estimator=xg_model,
    param_distributions=params,
    n_iter=5,
    cv=2,
    scoring=None,
    verbose=2,
    random_state=0
)

# Train
xgb_search.fit(X_train, y_train)

print("Best Parameters:")
print(xgb_search.best_params_)

print("\nBest Score:")
print(xgb_search.best_score_)

Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=  11.9s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=  10.9s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=  22.9s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=  26.9s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=  10.5s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=  10.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=200, subsample=0.8; total time=  11.4s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=200, subsample=0.8; total time=   8.4s
[CV] END col

In [11]:
import joblib
best_xgb = xgb_search.best_estimator_


xgb_pred = best_xgb.predict(X_test)


# Save model
joblib.dump(best_xgb, "../models/best_xgboost.pkl")

print("XGBoost model saved successfully!")
comparison = pd.DataFrame({"Actual_Sales": y_test.values,"Predicted_Sales": xgb_pred})

print(comparison.head(20))

XGBoost model saved successfully!
    Actual_Sales  Predicted_Sales
0       50932.42     58167.035156
1        3196.12      5822.192871
2       10125.03      9616.401367
3        3311.26      4851.236816
4        6335.65      4949.858398
5        8971.23      8947.821289
6        1575.35      4459.535645
7       17308.45     10311.681641
8        5763.39      7694.174316
9       17034.57     12961.940430
10        252.00      1650.781860
11       5927.00      4508.330566
12      23193.57     24807.449219
13      16863.99     16380.525391
14       6469.61      6669.010254
15      40789.48     47465.433594
16       1996.00      4230.056152
17       1451.39      7182.033691
18        795.89      4981.935059
19      13189.29     14201.773438


In [12]:
xgb_mae = mean_absolute_error(y_test, xgb_pred)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

xgb_r2 = r2_score(y_test, xgb_pred)

print("XG RMSE:", xgb_rmse)
print("XG MAE:", xgb_mae)
print("XG R2 Score:", xgb_r2)

XG RMSE: 7621.234127317287
XG MAE: 4590.568842570077
XG R2 Score: 0.8886166049182312
